# Fine-tuning LLaMA 3.1 8B on Alpaca Dataset

This notebook fine-tunes Meta's LLaMA 3.1 8B model on the Stanford Alpaca dataset using **QLoRA** (4-bit quantization + LoRA adapters).

After training, we merge the adapters and export to GGUF format for use with Ollama.

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import AutoPeftModelForCausalLM, LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Configuration

In [ ]:
# Model
MODEL_ID = "meta-llama/Llama-3.2-1B"  # requires HF access token with Meta approval

# Output
OUTPUT_DIR = "./model"
MERGED_MODEL_DIR = "./merged-model"

# Training hyperparameters
NUM_EPOCHS = 1
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4  # effective batch size = 4 * 4 = 16
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 512
WARMUP_RATIO = 0.03

# LoRA hyperparameters
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

## 4. Login to Hugging Face

You need a Hugging Face account with access to `meta-llama/Llama-3.1-8B`.  
1. Go to https://huggingface.co/meta-llama/Llama-3.1-8B and request access  
2. Create an access token at https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login

# Option 1: Interactive login (will prompt for token)
# login()

# Option 2: Direct token (uncomment and paste your token)
# 
login()

## 5. Load and Prepare the Alpaca Dataset

In [ ]:
dataset = load_dataset("tatsu-lab/alpaca", split="train")
print(f"Dataset size: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"\nExample:\n{dataset[0]}")

In [ ]:
def format_alpaca(example):
    """Format into LLaMA 3.1 chat template."""
    if example["input"].strip():
        user_message = f"{example['instruction']}\n\nInput:\n{example['input']}"
    else:
        user_message = example["instruction"]

    return {
        "text": (
            "<|begin_of_text|>"
            "<|start_header_id|>system<|end_header_id|>\n\n"
            "You are a helpful, respectful and honest assistant. "
            "Below is an instruction that describes a task. "
            "Write a response that appropriately completes the request."
            "<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n\n{user_message}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n{example['output']}<|eot_id|>"
        )
    }

dataset = dataset.map(format_alpaca)
print("Formatted example:")
print(dataset[0]["text"][:500])

## 6. Load Model with 4-bit Quantization

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded. Parameters: {model.num_parameters():,}")

## 7. Configure LoRA